In [1]:
# The client is already installed: docker-compose.yml mounts a script into the
# notebook image's before-notebook.d/, so confluent-kafka is there before this
# kernel exists. This cell is a check. If it fails, one line on your host:
#   docker exec -it firealarm-notebook pip install confluent-kafka==2.5.0
import confluent_kafka
print("confluent-kafka", confluent_kafka.version()[0])

confluent-kafka 2.5.0


In [2]:
from confluent_kafka.admin import AdminClient, NewTopic, NewPartitions
from confluent_kafka import KafkaException
import sys
from uuid import uuid4

In [3]:
bootstrap_server = "kafka:9092" # Brokers act as cluster entripoints

In [4]:
conf = {'bootstrap.servers': bootstrap_server}

In [5]:
a = AdminClient(conf)

In [6]:
md = a.list_topics(timeout=10)
print(" {} topics:".format(len(md.topics)))
for t in iter(md.topics.values()):
    if t.error is not None:
        errstr = ": {}".format(t.error)
    else:
        errstr = ""
    print("  \"{}\" with {} partition(s){}".format(t, len(t.partitions), errstr))

 0 topics:


In [7]:
from confluent_kafka import SerializingProducer
from confluent_kafka.serialization import *

import time

topic = "SmokeSensorEvent"

def delivery_report(err, msg):
    if err is not None:
        print("Failed to deliver message: {}".format(err))
    else:
        print("Produced record to topic {} partition [{}] @ offset {}"
              .format(msg.topic(), msg.partition(), msg.offset()))

In [8]:
producer_conf = {
        'bootstrap.servers': bootstrap_server,
        'key.serializer': StringSerializer('utf_8'),
        'value.serializer': StringSerializer('utf_8')
}

producer = SerializingProducer(producer_conf)

## run the following cell to send smoke event with `smoke==false`

In [9]:
import json
from IPython.display import clear_output

while True:
    key = "S1"
    value = {"sensor": "S1","smoke": False,"ts":int(time.time())}
    producer.produce(topic=topic, value=json.dumps(value), key=key, on_delivery=delivery_report)
    print(value)
    producer.poll(1)
    time.sleep(10)
    clear_output(wait=True)


{'sensor': 'S1', 'smoke': False, 'ts': 1789325625}
Produced record to topic SmokeSensorEvent partition [0] @ offset 12


KeyboardInterrupt: 

to interrupt the execution of the cell, prese the square icon in the bar or choose *interrupt kernel* from the *kernel* dropdown menu

## run the following cell to send smoke event with `smoke==true`

In [10]:
while True:
    key = "S1"
    value = {"sensor": "S1","smoke": True,"ts":int(time.time())}
    producer.produce(topic=topic, value=json.dumps(value), key=key, on_delivery=delivery_report)
    print(value)
    producer.poll(1)
    time.sleep(10)
    clear_output(wait=True)

{'sensor': 'S1', 'smoke': True, 'ts': 1789325824}
Produced record to topic SmokeSensorEvent partition [0] @ offset 32


KeyboardInterrupt: 